### Chunking

In [23]:
# Chunk by a set number of charactesr
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))

        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks

In [24]:
# Chunk by sentence
import re


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))

        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        start_idx += max_sentences_per_chunk - overlap_sentences

        if start_idx < 0:
            start_idx = 0

    return chunks

In [25]:
# Chunk by section
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [26]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

In [27]:
# chunk_by_char - 141
# chunk_by_sentence - 33
# chunk_by_section - 15 

In [28]:
len(chunks)

15

In [29]:
for chunk in chunks:
    print(chunk + "\n----\n")

# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

----

Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). **Financial Analysis** revealed mixed quarterly performance, prompting strategic reviews, particularly concerning resource allocation impacting R&D pipelines.

Crucial developments were also seen in **Scienti

### Embeddings

In [30]:
# Client Setup
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

In [31]:
# Embedding Generation
def generate_embedding(text, model="voyage-3-large", input_type="query"):
    result = client.embed([text], model=model, input_type=input_type)

    return result.embeddings[0] 

In [32]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

generate_embedding(chunks[0])

[-0.05453480780124664,
 0.01431055087596178,
 -0.016921261325478554,
 0.0005922440905123949,
 0.02136913500726223,
 0.037903621792793274,
 -0.047186143696308136,
 0.0024173229467123747,
 0.0025865354109555483,
 -0.01798488199710846,
 0.014020473696291447,
 0.048926617950201035,
 -0.00918582733720541,
 -0.0037710238248109818,
 -0.013730393722653389,
 0.007010236848145723,
 0.03326236084103584,
 0.08857070654630661,
 -0.04737953096628189,
 -0.010201102122664452,
 0.06033638119697571,
 -0.057628974318504333,
 0.06497763842344284,
 -0.03461606428027153,
 -0.018468346446752548,
 0.007638740353286266,
 0.007203621789813042,
 0.04486551135778427,
 -0.03248881921172142,
 -0.014407243579626083,
 -0.000906496134120971,
 0.022142676636576653,
 0.01885511912405491,
 0.05685543641448021,
 0.012473385781049728,
 0.04099779576063156,
 -0.06188346818089485,
 -0.020402204245328903,
 -0.02726740390062332,
 -0.054921574890613556,
 0.010587873868644238,
 0.005245590582489967,
 0.032682206481695175,
 -0.07

### Vector DB + semantic search

In [34]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [35]:
# VectorIndex implementation
import math 
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [36]:
# 1. Chunk by section
chunks = chunk_by_section(text)

# 2. Generate embeddings for each chunk
embeddings = generate_embedding(chunks)

In [37]:
# 3. Create a vector store and add each embedding to it
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):   
    store.add_vector(vector=embedding, document={"content": chunk}) 

In [44]:
# user_embedding = generate_embedding("What did the software engineering dept do last year?")
user_embedding = generate_embedding("What happened with INC-2023-Q4-011?")

In [45]:
result = store.search(user_embedding, k=3)
result

[({'content': 'Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011\n\nThe Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2023-Q4-011`. Threat intelligence indicates the activity aligns with tactics, techniques, and procedures associated with the `ShadowNet Syndicate` threat actor group. Initial access was gained via a spear-phishing email targeting personnel within the finance department, potentially seeking data relevant to Section 3 (Financial Analysis). Endpoint detection and response (EDR) systems flagged anomalous process execution (`PID: 7812`) on workstation `WS-FIN-112`. Subsequent investigation identified malware (`SHA256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`) attempting lateral movement towards server `SRV-FIN-03`. Containment involved isolating affected systems and blocking associated command-and-control infrastructure (IP `198.51.100.24`). Mitigation in

In [46]:
for doc, distance in result:
    print(f"Distance: {distance:.4f}\nContent: {doc['content']}\n----\n")

Distance: 0.4832
Content: Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2023-Q4-011`. Threat intelligence indicates the activity aligns with tactics, techniques, and procedures associated with the `ShadowNet Syndicate` threat actor group. Initial access was gained via a spear-phishing email targeting personnel within the finance department, potentially seeking data relevant to Section 3 (Financial Analysis). Endpoint detection and response (EDR) systems flagged anomalous process execution (`PID: 7812`) on workstation `WS-FIN-112`. Subsequent investigation identified malware (`SHA256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`) attempting lateral movement towards server `SRV-FIN-03`. Containment involved isolating affected systems and blocking associated command-and-control infrastructure (IP `198.51.100.24`). Miti

### BM25 lexical search

In [56]:
# Step - 1: Tokenize the query 
# Step - 2: Count the frequency: See how often each term appears in all your documents. 
# Step - 3: Weight terms by importance: Assign higher weights to terms that are less frequent and less weight to common terms. 
# Step - 4: Find best matches: Return documents that contains more instances of the higher-weighted terms. 

In [57]:
# BM25Index implementation
import math
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def _compute_bm25_score(
        self, query_tokens: List[str], doc_index: int
    ) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query_text: str,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

In [58]:
# 1. Chunk by section
chunks = chunk_by_section(text)

# 2. Generate embeddings for each chunk
embeddings = generate_embedding(chunks)

In [59]:
len(embeddings), len(chunks)

(15, 15)

In [60]:
# Create BM25 store and add each chunk to it 
store = BM25Index()

for chunk in chunks:
    store.add_document({"content": chunk})

In [61]:
# Search the store with a user query 
results = store.search("What happened with INC-2023-Q4-011?", 3)

In [62]:
results

[({'content': 'Section 2: Software Engineering - Project Phoenix Stability Enhancements\n\nThe Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies. The deployment of a patch addressed the memory allocation error, resulting in a measured 40% reduction in critical failures under simulated stress tests during Q4 2024 (Test Case ID: INC-2023-Q4-011). Further refactoring of the query module, scheduled for the next release cycle, aims to resolve the timeout issue. These findings underscore the importance of robust testing protocols, especially given the dependencies

In [63]:
for doc, dist in result:
    print(f"Distance: {dist:.4f}\nContent: {doc['content']}\n----\n")

Distance: 0.2712
Content: Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies. The deployment of a patch addressed the memory allocation error, resulting in a measured 40% reduction in critical failures under simulated stress tests during Q4 2024 (Test Case ID: INC-2023-Q4-011). Further refactoring of the query module, scheduled for the next release cycle, aims to resolve the timeout issue. These findings underscore the importance of robust testing protocols, especially given the dep

In [64]:
# BM25 excels at finding exact matches because it:
#     Gives higher weight to rare, specific terms
#     Ignores common words that don't add search value
#     Focuses on term frequency rather than semantic meaning
#     Works especially well for technical terms, IDs, and specific phrases

### Multi-Index RAG pipeline: Semantic + Lexical Search